# XGBoost Baseline — QSO

Referencia para [../05_xgboost_optuna/xgb_optuna_qso.ipynb](../05_xgboost_optuna/xgb_optuna_qso.ipynb).


## 1. Setup

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "config.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config import paths_for, RESULTS_DIR as PROJECT_RESULTS, MODELS_DIR, print_info

OBJECT_TYPE = "QSO"
print_info()

paths = paths_for(OBJECT_TYPE)
HDF5_PATH = paths["spectra_h5"].with_name(f"{OBJECT_TYPE}spectra_padded.h5")
print(f"\nObjeto: {OBJECT_TYPE}")
print(f"HDF5:   {HDF5_PATH}")


In [ ]:
import numpy as np, pandas as pd

from src.data import load_spectral_dataset, normalize_spectra, make_or_load_split, apply_split
from config import SPLITS_DIR
from src.models import train_xgboost
from src.evaluation import compute_redshift_metrics

SEED = 42
RESULTS_DIR = PROJECT_RESULTS / OBJECT_TYPE / "xgboost_baseline"
MODEL_DIR   = MODELS_DIR / OBJECT_TYPE / "xgboost_baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True); MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 2. Dados

In [ ]:
X, y, _ = load_spectral_dataset(HDF5_PATH, seed=SEED)
X = normalize_spectra(X)
# Split canonico estratificado por z (mesmo de todos os modelos).
tr, va, te = make_or_load_split(OBJECT_TYPE, y, SPLITS_DIR)
X_train, X_val, X_test = apply_split(X, tr, va, te)
y_train, y_val, y_test = apply_split(y, tr, va, te)


## 3. Treinar

In [ ]:
model = train_xgboost(X_train, y_train, X_val, y_val, seed=SEED)
y_pred = model.predict(X_test)
results = compute_redshift_metrics(y_test, y_pred)
print(f"RMSE: {results['rmse']:.4f}  R2: {results['r2']:.4f}  NMAD: {results['nmad']:.4f}")


## 4. Salvar

In [ ]:
# Fix xgboost 3.x + sklearn 1.8: save_model() checa model._estimator_type,
# que o sklearn 1.8 removeu do RegressorMixin. Definimos manualmente.
model._estimator_type = "regressor"
model.save_model(MODEL_DIR / "xgboost_baseline.json")
np.savez(RESULTS_DIR / "predictions.npz", y_test=y_test, y_pred=y_pred, delta_z=results["delta_z"])
pd.DataFrame([{"model": "xgboost_baseline", "object": OBJECT_TYPE,
               **{k: v for k, v in results.items() if isinstance(v, (int, float))}}
            ]).to_csv(RESULTS_DIR / "metrics.csv", index=False)
